In [1]:
!pip install transformers

  Using cached transformers-4.55.2-py3-none-any.whl.metadata (41 kB)
  Using cached huggingface_hub-0.34.4-py3-none-any.whl.metadata (14 kB)
  Using cached tokenizers-0.21.4-cp39-abi3-win_amd64.whl.metadata (6.9 kB)
  Using cached safetensors-0.6.2-cp38-abi3-win_amd64.whl.metadata (4.1 kB)
Using cached transformers-4.55.2-py3-none-any.whl (11.3 MB)
Using cached huggingface_hub-0.34.4-py3-none-any.whl (561 kB)
Using cached tokenizers-0.21.4-cp39-abi3-win_amd64.whl (2.5 MB)
Using cached safetensors-0.6.2-cp38-abi3-win_amd64.whl (320 kB)

   ---------------- ----------------------- 2/5 [huggingface-hub]
   ---------------- ----------------------- 2/5 [huggingface-hub]
   ---------------- ----------------------- 2/5 [huggingface-hub]
   ------------------------ --------------- 3/5 [tokenizers]
   -------------------------------- ------- 4/5 [transformers]
   -------------------------------- ------- 4/5 [transformers]
   -------------------------------- ------- 4/5 [transformers]
   -------

In [2]:
# Use a pipeline as a high-level helper
from transformers import pipeline

pipe = pipeline("image-classification", model="prithivMLmods/Face-Mask-Detection")
pipe("https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/hub/parrots.png")

c:\Users\sonng\miniconda3\envs\autovrs\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
Device set to use cpu


[{'label': 'Face_Mask Found', 'score': 0.9183592200279236},
 {'label': 'Face_Mask Not_Found', 'score': 0.08164083957672119}]

In [ ]:
import cv2
from transformers import pipeline
import numpy as np

# Initialize the pipeline
pipe = pipeline("image-classification", model="prithivMLmods/Face-Mask-Detection")

# Start webcam
cap = cv2.VideoCapture(0)
if not cap.isOpened():
    print("Cannot open camera")
    exit()

# Process webcam frames
print("Starting webcam, press 'q' to quit...")
while True:
    ret, frame = cap.read()
    if not ret:
        print("Failed to grab frame")
        break
    
    # Convert frame to RGB
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    
    # Run the pipeline
    results = pipe(rgb_frame)
    
    # Display results on the frame
    for result in results:
        label = result['label']
        score = result['score']
        text = f"{label}: {score:.2f}"
        cv2.putText(frame, text, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
    
    cv2.imshow("Mask Detection", frame)
    
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break
cap.release()
cv2.destroyAllWindows()

Device set to use cpu


Cannot open camera
Starting webcam, press 'q' to quit...
Failed to grab frame


: 

In [ ]:
from transformers import pipeline, AutoModelForImageClassification, AutoTokenizer
import torch
from onnxruntime import InferenceSession
import numpy as np
import cv2
import os

# Load the model and tokenizer
model_name = "prithivMLmods/Face-Mask-Detection"
model = AutoModelForImageClassification.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Convert to ONNX
onnx_model_path = "face_mask_detection.onnx"
if not os.path.exists(onnx_model_path):
    dummy_input = torch.randn(1, 3, 224, 224)  # Assuming input size is (3, 224, 224)
    torch.onnx.export(model, dummy_input, onnx_model_path,
                      input_names=["input"],
                      output_names=["output"],
                      dynamic_axes={"input": {0: "batch_size"}, "output": {0: "batch_size"}},
                      opset_version=11)

# Load ONNX model with ONNX Runtime
session = InferenceSession(onnx_model_path)

# Start webcam
cap = cv2.VideoCapture(0)
if not cap.isOpened():
    print("Cannot open camera")
    exit()

print("Starting webcam, press 'q' to quit...")
while True:
    ret, frame = cap.read()
    if not ret:
        print("Failed to grab frame")
        break

    # Preprocess frame
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    resized_frame = cv2.resize(rgb_frame, (224, 224))
    input_data = np.transpose(resized_frame, (2, 0, 1)).astype(np.float32) / 255.0
    input_data = np.expand_dims(input_data, axis=0)

    # Run inference
    inputs = {session.get_inputs()[0].name: input_data}
    outputs = session.run(None, inputs)
    predictions = outputs[0]

    # Display results
    label = np.argmax(predictions)
    score = np.max(predictions)
    text = f"Label: {label}, Score: {score:.2f}"
    cv2.putText(frame, text, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
    cv2.imshow("Mask Detection", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break
cap.release()
cv2.destroyAllWindows()